In [17]:
!sudo apt-get install python3.13 python3.13-dev libpython3.13-dev -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpython3.13 libpython3.13-stdlib
Suggested packages:
  python3.13-venv
The following NEW packages will be installed:
  libpython3.13 libpython3.13-dev libpython3.13-stdlib python3.13
  python3.13-dev
0 upgraded, 5 newly installed, 0 to remove and 63 not upgraded.
Need to get 14.3 MB of archives.
After this operation, 56.1 MB of additional disk space will be used.
Get:1 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 libpython3.13-stdlib amd64 3.13.12-1+jammy1 [3,000 kB]
Get:2 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 libpython3.13 amd64 3.13.12-1+jammy1 [2,290 kB]
Get:3 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 libpython3.13-dev amd64 3.13.12-1+jammy1 [5,633 kB]
Get:4 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 python3.13 am

In [18]:
!sudo apt-get update -y

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 https://cli.github.com/packages stable InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,477 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,903 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,717 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,300 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,745 kB]
Get:14 

In [19]:
# Install required libraries.
# We use specific versions of tensorflow and shap to ensure compatibility.
# Install required libraries.
# We use specific versions of tensorflow and shap to ensure compatibility.
# We also pin numpy<2.0 to prevent breaking changes that affect the shap library.
!pip install pandas optuna scikit-learn==1.8.0 tensorflow shap numpy xgboost ml_dtypes JAX scikit-learn-intelex

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
zip_file_path = "/content/drive/MyDrive/UNS/2023/Business Intelligence/Paper/Salomo/NewIJAI/Archive 4.zip"

In [22]:
!rm -rf /content/config
!rm -rf /content/Dataset
!rm -rf /content/src

!rm -rf /content/NewIJAI
# Unzip the project into the /content/ directory
!unzip -o -q "{zip_file_path}" -d "/content/NewIJAI"


print("Project unzipped successfully!")


shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Project unzipped successfully!


In [23]:
!ls -l

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
total 0


In [24]:
import sys
import os

# --- IMPORTANT ---
# If your unzipped project folder is not named 'NewIJAI', change it here.
# This should be the root folder that contains your 'src' directory.
project_root = '/content/NewIJAI'
src_path = os.path.join(project_root, 'src')

if src_path not in sys.path:
    sys.path.insert(0, src_path)
    print(f"Added '{src_path}' to Python path.")

# Change the current working directory to the project root
os.chdir(project_root)
print(f"Changed working directory to: {os.getcwd()}")

# Verify that your project files are visible
!ls -F


Changed working directory to: /content/NewIJAI
config/  Dataset/  __MACOSX/  model_works_refactored.ipynb  src/


In [25]:
import sklearn
print(sklearn.__version__)

import warnings
warnings.filterwarnings("ignore")
# Other plots (example)

from src.exp import (
    ExperimentConfig, ExperimentFacade,
    DataReadConfig, PlotManager
)

1.8.0


In [26]:
try:
    from sklearnex import patch_sklearn
    patch_sklearn()
    print("Intel sklearn patch enabled")
except ImportError:
    print("sklearnex not installed; using standard sklearn")


Intel sklearn patch enabled


Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


In [27]:
data_cfg = DataReadConfig(
    root_dir="Dataset/data",
    recursive=True,
    exclude_filenames=["cclass.csv", "unclean focus.csv","unclean cclass.csv","focus.csv"],  # the excluded files
    add_source_column=False,        # enable this to make additional column filled with the original taken filenames
)

In [28]:
cfg = ExperimentConfig(
    outer_folds=5,
    inner_folds=5,
    n_trials=40,
    seed=42,
    log_target=True
)

In [29]:
models = ["LinearRegression", "DecisionTree", "RandomForest", "XGBoost", "NeuralNetwork"]

In [30]:
exp = ExperimentFacade.from_folder(
    data_cfg=data_cfg,
    target="price",
    cfg=cfg,
    model_names=models,
    hparam_json="config/hyperparams.json"
)

# model, year, price, transmission, mileage, fuelType, tax, mpg, engineSize

[schema]
  numerical cols: ['year', 'mileage', 'tax', 'mpg', 'engineSize']
  categorical cols: ['model', 'transmission', 'fuelType']
  target col: ['price']
  mapping: {'model': 'model', 'year': 'year', 'price': 'price', 'transmission': 'transmission', 'mileage': 'mileage', 'fuelType': 'fuelType', 'tax': 'tax', 'mpg': 'mpg', 'engineSize': 'engineSize'}


In [ ]:
results = exp.run()

[I 2026-02-17 00:51:58,328] A new study created in memory with name: LinearRegression_OuterFold_1_residual_cfg_base
[I 2026-02-17 00:52:03,922] Trial 0 finished with value: 2119.603067136768 and parameters: {}. Best is trial 0 with value: 2119.603067136768.
[I 2026-02-17 00:52:04,009] Trial 1 finished with value: 2119.603067136768 and parameters: {}. Best is trial 0 with value: 2119.603067136768.
[I 2026-02-17 00:52:04,072] Trial 2 finished with value: 2119.603067136768 and parameters: {}. Best is trial 0 with value: 2119.603067136768.
[I 2026-02-17 00:52:04,129] Trial 3 finished with value: 2119.603067136768 and parameters: {}. Best is trial 0 with value: 2119.603067136768.
[I 2026-02-17 00:52:04,183] Trial 4 finished with value: 2119.603067136768 and parameters: {}. Best is trial 0 with value: 2119.603067136768.
[I 2026-02-17 00:52:04,240] Trial 5 finished with value: 2119.603067136768 and parameters: {}. Best is trial 0 with value: 2119.603067136768.
[I 2026-02-17 00:52:05,386] A ne

496/496 ━━━━━━━━━━━━━━━━━━━━ 0s 717us/step
496/496 ━━━━━━━━━━━━━━━━━━━━ 0s 725us/step
496/496 ━━━━━━━━━━━━━━━━━━━━ 0s 783us/step
496/496 ━━━━━━━━━━━━━━━━━━━━ 0s 737us/step
496/496 ━━━━━━━━━━━━━━━━━━━━ 0s 750us/step


[I 2026-02-17 01:51:49,386] Trial 0 finished with value: 1954.328818820254 and parameters: {'optimizer': 'adamW', 'n_layers': 2, 'units_layer_1': 93, 'units_layer_2': 182, 'units_layer_3': 120, 'units_layer_4': 95, 'activation': 'relu', 'learning_rate': 0.008912031574631267, 'momentum': 0.4442189556872475, 'weight_decay': 2.6333841844193604e-06}. Best is trial 0 with value: 1954.328818820254.


496/496 ━━━━━━━━━━━━━━━━━━━━ 0s 832us/step
496/496 ━━━━━━━━━━━━━━━━━━━━ 0s 846us/step
496/496 ━━━━━━━━━━━━━━━━━━━━ 0s 824us/step
496/496 ━━━━━━━━━━━━━━━━━━━━ 0s 866us/step
496/496 ━━━━━━━━━━━━━━━━━━━━ 0s 879us/step


[I 2026-02-17 01:58:27,790] Trial 1 finished with value: 2083.1201911246662 and parameters: {'optimizer': 'sgd', 'n_layers': 3, 'units_layer_1': 90, 'units_layer_2': 156, 'units_layer_3': 123, 'units_layer_4': 63, 'activation': 'sigmoid', 'learning_rate': 0.009308155018643535, 'momentum': 0.21940831436629465, 'weight_decay': 1.955409609247652e-06}. Best is trial 0 with value: 1954.328818820254.


...

[I 2026-02-17 05:52:57,858] Trial 9 finished with value: 1910.1595546348121 and parameters: {'optimizer': 'adam', 'n_layers': 3, 'units_layer_1': 41, 'units_layer_2': 76, 'units_layer_3': 78, 'units_layer_4': 222, 'activation': 'relu', 'learning_rate': 0.00015363635484309001, 'momentum': 0.8228903695802072, 'weight_decay': 2.0881026313344575e-05}. Best is trial 2 with value: 1821.3818125883488.


496/496 ━━━━━━━━━━━━━━━━━━━━ 1s 945us/step


[I 2026-02-17 05:54:21,918] Trial 10 finished with value: inf and parameters: {'optimizer': 'sgd', 'n_layers': 2, 'units_layer_1': 247, 'units_layer_2': 171, 'units_layer_3': 196, 'units_layer_4': 41, 'activation': 'relu', 'learning_rate': 0.009445558171352885, 'momentum': 0.28467406032622306, 'weight_decay': 4.598761173099933e-06}. Best is trial 2 with value: 1821.3818125883488.


620/620 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


[I 2026-02-17 05:56:18,535] A new study created in memory with name: LinearRegression_OuterFold_3_residual_cfg_base
[I 2026-02-17 05:56:23,350] Trial 0 finished with value: 2064.2094191737615 and parameters: {}. Best is trial 0 with value: 2064.2094191737615.
[I 2026-02-17 05:56:23,426] Trial 1 finished with value: 2064.2094191737615 and parameters: {}. Best is trial 0 with value: 2064.2094191737615.
[I 2026-02-17 05:56:23,480] Trial 2 finished with value: 2064.2094191737615 and parameters: {}. Best is trial 0 with value: 2064.2094191737615.
[I 2026-02-17 05:56:23,533] Trial 3 finished with value: 2064.2094191737615 and parameters: {}. Best is trial 0 with value: 2064.2094191737615.
[I 2026-02-17 05:56:23,586] Trial 4 finished with value: 2064.2094191737615 and parameters: {}. Best is trial 0 with value: 2064.2094191737615.
[I 2026-02-17 05:56:23,644] Trial 5 finished with value: 2064.2094191737615 and parameters: {}. Best is trial 0 with value: 2064.2094191737615.
[I 2026-02-17 05:56:

In [ ]:
display(exp.summary())
model_summary = exp.summary()["model"]

In [ ]:
model_summary.to_list()

In [ ]:
sig = exp.significance(
    metric="R2",
    baseline="NeuralNetwork",
    models=model_summary.to_list()
)

In [ ]:
display(sig)

In [ ]:
shap_analyzer = exp.shap(models=model_summary.to_list())

In [ ]:
shap_analyzer.available_models()

In [ ]:
for m in shap_analyzer.available_models():
    shap_analyzer.beeswarm(m)

In [ ]:
exp.summary()

In [ ]:
results.to_csv("outputs/experiment_results.csv", index=False)

In [ ]:
pm = PlotManager("outputs/figures/metrics")

In [ ]:
for metric in ["R2", "MAE", "MedAE", "MSE", "RMSE"]:
    is_ascending = metric != "R2" and metric != "NegMSE"
    #fig = plot_point_range_1(results, metric=metric, ascending=is_ascending)
    #fig.savefig(f"outputs/figures/metrics/point_range_{metric.lower()}", dpi=300, bbox_inches="tight")
    fig = pm.plot_point_range(results_df=results, metric=metric, ascending=is_ascending)
    pm.save_fig(fig, f"point_range_{metric.lower()}")

In [ ]:
for metric in ["R2", "MAE", "MedAE", "MSE", "RMSE"]:
    display(exp.significance_matrix(metric=metric))

In [ ]:
exp.save_best_params()